In [33]:
import requests
from bs4 import BeautifulSoup

Response = requests.get("https://dealerbyd.id/alamat-dealer-byd/")
print(Response.status_code)
#print(Response.text)

200


In [34]:
soup = BeautifulSoup(Response.text, 'html.parser')
table_blocks = soup.find_all('div', class_='elementor-element elementor-element-f816f01 e-con-full e-flex e-con e-child')
print("Number of table found: ", len(table_blocks))

Number of table found:  1


In [35]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# --- Request ---
url = "https://dealerbyd.id/alamat-dealer-byd/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36"
}

print("Mengambil data dari website...")
response = requests.get(url, headers=headers, timeout=15)
response.raise_for_status()
print(f"Status: {response.status_code} - Berhasil!")

soup = BeautifulSoup(response.text, 'html.parser')

# --- Ekstrak Nama Dealer & Alamat ---
result = []

# Cari semua heading dealer (h2 dengan link)
dealer_headings = soup.find_all('h2', class_='elementor-heading-title')

print(f"Mencari {len(dealer_headings)} dealer...")

for heading in dealer_headings:
    try:
        # --- Nama Dealer ---
        name_tag = heading.find('a')
        dealer_name = name_tag.get_text(strip=True) if name_tag else "N/A"

        # --- Alamat: cari di widget icon-list setelah heading ---
        # Naik ke parent container, lalu cari sibling berikutnya yang berisi alamat
        parent_container = heading.find_parent('div', class_='elementor-element')
        if not parent_container:
            continue

        # Cari widget icon-list di sibling berikutnya
        icon_list_widget = parent_container.find_next_sibling('div', class_='elementor-widget-icon-list')
        if not icon_list_widget:
            # Jika tidak ada sibling, coba cari di dalam parent atau setelahnya
            icon_list_widget = parent_container.find_next('div', class_='elementor-widget-icon-list')

        address = "Tidak tersedia"
        if icon_list_widget:
            text_span = icon_list_widget.find('span', class_='elementor-icon-list-text')
            if text_span:
                address = text_span.get_text(strip=True).replace('\n', ' ').strip()

        # Simpan
        result.append({
            'Dealer': dealer_name,
            'Alamat': address
        })

    except Exception as e:
        print(f"Error memproses dealer: {e}")
        continue

# --- Buat DataFrame ---
print(f"\nData berhasil diekstrak untuk {len(result)} dealer...")

df = pd.DataFrame(result)

print(f"\nPreview data:")
print(df.head(10))
print(f"\nNote: Excel akan disimpan setelah scraping koordinat selesai.")


Mengambil data dari website...
Status: 200 - Berhasil!
Mencari 57 dealer...

Data berhasil diekstrak untuk 57 dealer...

Preview data:
                  Dealer                                             Alamat
0              BYD Medan  Jl. T. Amir Hamzah Kel. Sei Agul, Kec. Medan B...
1  BYD Bekasi Summarecon  Jl. Bulevar Utara Ahmad Yani KB/008 No.55, RT....
2       BYD Bekasi Timur  Jl. Diponegoro No.2, Jatimulya, Kec. Tambun Se...
3    BYD Arista BSD City  Kav Komercial The Park, Kav 1.7 Jl. BSD Raya U...
4            BYD Cirebon  Jl. DR. Cipto Mangunkusumo No.155a, Pekiringan...
5              BYD Depok  Jl. Margonda Raya No. 888 Kel. Kemirimuka, Kec...
6         BYD Daan Mogot  Jl. Daan Mogot KM. 11 No.3a, RT.12/RW.3, Cengk...
7  BYD Arista Kalimalang  Jl. Raya Kalimalang No. 38 Kel. Duren Sawit, K...
8                    N/A  Jl. Gatot Subroto No.Kav 40-41, RT.8/RW.3, Kun...
9              BYD Tebet  Jl. Dr. Saharjo No.246, Menteng Dalam, Kec. Te...

Note: Excel akan disimpan se

In [ ]:
import pandas as pd
import os
import time
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# --- Gunakan DataFrame dari Cell 2 ---
# Pastikan Cell 2 sudah dijalankan untuk membuat variabel 'df'
if 'df' not in globals():
    raise NameError("DataFrame 'df' tidak ditemukan! Silakan jalankan Cell 2 terlebih dahulu.")

print(f"✓ Menggunakan DataFrame dari Cell 2")
print(f"Total dealer: {len(df)}")
print(f"\nPreview data:")
print(df.head())

# Inisialisasi kolom Latitude dan Longitude jika belum ada
if 'Latitude' not in df.columns:
    df['Latitude'] = None
if 'Longitude' not in df.columns:
    df['Longitude'] = None


Total dealer: 57

Preview data:
                  Dealer                                             Alamat
0              BYD Medan  Jl. T. Amir Hamzah Kel. Sei Agul, Kec. Medan B...
1  BYD Bekasi Summarecon  Jl. Bulevar Utara Ahmad Yani KB/008 No.55, RT....
2       BYD Bekasi Timur  Jl. Diponegoro No.2, Jatimulya, Kec. Tambun Se...
3    BYD Arista BSD City  Kav Komercial The Park, Kav 1.7 Jl. BSD Raya U...
4            BYD Cirebon  Jl. DR. Cipto Mangunkusumo No.155a, Pekiringan...


In [37]:
# --- Setup Selenium WebDriver ---
print("\nMembuka browser...")
options = webdriver.ChromeOptions()
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(options=options)
driver.maximize_window()

# Counter untuk progress
total_dealers = len(df)
processed = 0

print(f"\nMemulai scraping koordinat untuk {total_dealers} dealer...")
print("=" * 60)

try:
    for index, row in df.iterrows():
        dealer_name = str(row['Dealer']).strip()
        
        # Skip jika dealer N/A atau sudah ada koordinat
        if dealer_name == 'N/A' or pd.notna(row['Latitude']):
            print(f"[{index+1}/{total_dealers}] SKIP: {dealer_name}")
            continue
        
        print(f"\n[{index+1}/{total_dealers}] Mencari koordinat: {dealer_name}")
        
        try:
            # --- Cari di Google Maps ---
            # Format URL: google.com/maps/search/[query]
            search_query = f"{dealer_name} Indonesia"
            maps_url = f"https://www.google.com/maps/search/{search_query.replace(' ', '+')}"
            
            driver.get(maps_url)
            time.sleep(3)  # Tunggu page load
            
            # --- Ambil koordinat dari URL ---
            # URL Maps biasanya: .../@lat,lng,zoom... atau .../@lat,lng,zoom/data=...
            current_url = driver.current_url
            print(f"URL Maps: {current_url[:100]}...")
            
            # Extract koordinat dari URL menggunakan regex
            # Format: @latitude,longitude,zoom
            coord_pattern = r'@(-?\d+\.\d+),(-?\d+\.\d+),'
            match = re.search(coord_pattern, current_url)
            
            if match:
                latitude = float(match.group(1))
                longitude = float(match.group(2))
                
                # Simpan ke dataframe
                df.at[index, 'Latitude'] = latitude
                df.at[index, 'Longitude'] = longitude
                
                print(f"✓ Koordinat ditemukan: Lat {latitude}, Long {longitude}")
                processed += 1
            else:
                # Alternatif: coba ambil dari URL dengan format berbeda
                # Atau klik link pertama untuk mendapatkan koordinat
                try:
                    # Tunggu search results muncul
                    wait = WebDriverWait(driver, 10)
                    first_result = wait.until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='@']"))
                    )
                    first_result.click()
                    time.sleep(2)
                    
                    # Ambil URL setelah klik
                    new_url = driver.current_url
                    match = re.search(coord_pattern, new_url)
                    
                    if match:
                        latitude = float(match.group(1))
                        longitude = float(match.group(2))
                        
                        df.at[index, 'Latitude'] = latitude
                        df.at[index, 'Longitude'] = longitude
                        
                        print(f"✓ Koordinat ditemukan (setelah klik): Lat {latitude}, Long {longitude}")
                        processed += 1
                    else:
                        print(f"✗ Tidak dapat menemukan koordinat di URL")
                except (TimeoutException, NoSuchElementException) as e:
                    print(f"✗ Error: Tidak dapat mengakses hasil Maps - {str(e)}")
            
        except Exception as e:
            print(f"✗ Error memproses {dealer_name}: {str(e)}")
            continue
        
        # Delay antar request untuk menghindari rate limiting
        time.sleep(2)
    
    print("\n" + "=" * 60)
    print(f"Scraping selesai! Berhasil mendapatkan {processed} koordinat.")
    
    # --- Simpan ke Excel ---
    output_file = 'Daftar_Dealer_BYD_Indonesia.xlsx'
    output_path = os.path.join(os.getcwd(), output_file)
    df.to_excel(output_path, index=False, engine='openpyxl')
    print(f"\nFile Excel berhasil disimpan di: {output_path}")
    
    # Tampilkan summary
    print(f"\nSummary:")
    print(f"- Total dealer: {total_dealers}")
    print(f"- Koordinat ditemukan: {df['Latitude'].notna().sum()}")
    print(f"- Koordinat belum ditemukan: {df['Latitude'].isna().sum()}")
    
except KeyboardInterrupt:
    print("\n\nProses dihentikan oleh user. Menyimpan progress...")
    output_file = 'Daftar_Dealer_BYD_Indonesia.xlsx'
    output_path = os.path.join(os.getcwd(), output_file)
    df.to_excel(output_path, index=False, engine='openpyxl')
    print(f"Progress tersimpan di: {output_path}")
    
finally:
    driver.quit()
    print("\nBrowser ditutup.")


Membuka browser...

Memulai scraping koordinat untuk 57 dealer...

[1/57] Mencari koordinat: BYD Medan
URL Maps: https://www.google.com/maps/place/BYD+Arista+Amir+Hamzah/@3.6071654,98.660692,17z/data=!3m1!4b1!4m6!...
✓ Koordinat ditemukan: Lat 3.6071654, Long 98.660692

[2/57] Mencari koordinat: BYD Bekasi Summarecon
URL Maps: https://www.google.com/maps/search/BYD+Bekasi+Summarecon+Indonesia...
✗ Error: Tidak dapat mengakses hasil Maps - Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff770127a35
	0x7ff770127a90
	0x7ff76fea16ad
	0x7ff76fefa13e
	0x7ff76fefa44c
	0x7ff76ff4ebe7
	0x7ff76ff4b8fb
	0x7ff76feeb068
	0x7ff76feebe93
	0x7ff7703e29d0
	0x7ff7703dce50
	0x7ff7703fcc45
	0x7ff7701430ce
	0x7ff77014adbf
	0x7ff770130c14
	0x7ff770130dcf
	0x7ff770116828
	0x7fff3ecfe8d7
	0x7fff3fccc53c


[3/57] Mencari koordinat: BYD Bekasi Timur
URL Maps: https://www.google.com/maps/search/BYD+Bekasi+Timur+Indonesia...
✗ Error: Tidak dapat mengakses hasil Maps - Message: 
Sta